# Zadanie B — Model wielojęzyczny (mBERT)

In [1]:
import transformers
import inspect
from transformers import Trainer

print("transformers version:", transformers.__version__)
print("Trainer from:", Trainer.__module__)
print("Trainer init signature:", inspect.signature(Trainer.__init__))

/Users/timbarvenov/Documents/uam/sem1/uczenie_glebokie_all/uczenie_glebokie/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers version: 4.57.3
Trainer from: transformers.trainer
Trainer init signature: (self, model: Union[transformers.modeling_utils.PreTrainedModel, torch.nn.modules.module.Module, NoneType] = None, args: Optional[transformers.training_args.TrainingArguments] = None, data_collator: Optional[Callable[[list[Any]], dict[str, Any]]] = None, train_dataset: Union[torch.utils.data.dataset.Dataset, torch.utils.data.dataset.IterableDataset, ForwardRef('datasets.Dataset'), NoneType] = None, eval_dataset: Union[torch.utils.data.dataset.Dataset, dict[str, torch.utils.data.dataset.Dataset], ForwardRef('datasets.Dataset'), NoneType] = None, processing_class: Union[transformers.tokenization_utils_base.PreTrainedTokenizerBase, transformers.image_processing_utils.BaseImageProcessor, transformers.feature_extraction_utils.FeatureExtractionMixin, transformers.processing_utils.ProcessorMixin, NoneType] = None, model_init: Optional[Callable[..., transformers.modeling_utils.PreTrainedModel]] = None, comp

In [1]:
import os

In [3]:
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

In [4]:
import numpy as np
import torch
from datasets import load_dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

device = 'mps'

/Users/timbarvenov/Documents/uam/sem1/uczenie_glebokie_all/uczenie_glebokie/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
ds = load_dataset("glue", "sst2")

train_small = ds["train"].shuffle(seed=42).select(range(5000))
val_small   = ds["validation"].shuffle(seed=42).select(range(500))

train_small, val_small

(Dataset({
     features: ['sentence', 'label', 'idx'],
     num_rows: 5000
 }),
 Dataset({
     features: ['sentence', 'label', 'idx'],
     num_rows: 500
 }))

In [6]:
model_name = "bert-base-multilingual-cased"  # mBERT
tokenizer = AutoTokenizer.from_pretrained(model_name)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [7]:
def preprocess(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=128)

train_small_tok = train_small.map(preprocess, batched=True)
val_small_tok   = val_small.map(preprocess, batched=True)

Map: 100%|██████████| 5000/5000 [00:00<00:00, 29843.44 examples/s]


In [8]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

metric_acc = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric_acc.compute(predictions=preds, references=labels)

id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}
model.config.id2label = id2label
model.config.label2id = label2id

model.to(device);

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
args = TrainingArguments(
    output_dir="./mbert_sst2_small",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_small_tok,
    eval_dataset=val_small_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/var/folders/1r/k8ndmm417js8lrym5jjwrzp80000gn/T/ipykernel_55584/3119133880.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/Users/timbarvenov/Documents/uam/sem1/uczenie_glebokie_all/uczenie_glebokie/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.615100
100,0.503900
150,0.441300


TrainOutput(global_step=157, training_loss=0.5163337166901607, metrics={'train_runtime': 132.6533, 'train_samples_per_second': 37.692, 'train_steps_per_second': 1.184, 'total_flos': 68992652047680.0, 'train_loss': 0.5163337166901607, 'epoch': 1.0})

In [10]:
import torch.nn.functional as F

def predict_texts(texts):
    enc = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**enc)
        probs = F.softmax(out.logits, dim=-1).detach().cpu().numpy()
    preds = probs.argmax(axis=-1)
    return preds, probs

val = ds["validation"]
idxs = [0, 3, 10, 25, 40, 55]
en_texts = [val[i]["sentence"] for i in idxs]
en_labels = [val[i]["label"] for i in idxs]

en_preds, en_probs = predict_texts(en_texts)

for t, y, p, pr in zip(en_texts, en_labels, en_preds, en_probs):
    print("EN:", t)
    print("  gold:", id2label[y], " pred:", id2label[int(p)], " prob:", {id2label[i]: float(pr[i]) for i in range(2)})
    print()

EN: it 's a charming and often affecting journey . 
  gold: positive  pred: positive  prob: {'negative': 0.06462125480175018, 'positive': 0.9353787899017334}

EN: the acting , costumes , music , cinematography and sound are all astounding given the production 's austere locales . 
  gold: positive  pred: positive  prob: {'negative': 0.14902694523334503, 'positive': 0.8509730100631714}

EN: the mesmerizing performances of the leads keep the film grounded and keep the audience riveted . 
  gold: positive  pred: positive  prob: {'negative': 0.07739045470952988, 'positive': 0.9226095676422119}

EN: nicks , seemingly uncertain what 's going to make people laugh , runs the gamut from stale parody to raunchy sex gags to formula romantic comedy . 
  gold: negative  pred: negative  prob: {'negative': 0.7540048956871033, 'positive': 0.24599511921405792}

EN: there 's ... tremendous energy from the cast , a sense of playfulness and excitement that seems appropriate . 
  gold: positive  pred: posi

In [11]:
# polish translations
pl_texts = [
    "(PL) To urocza i często poruszająca podróż.",
    "(PL) Gra aktorska, kostiumy, muzyka, zdjęcia i dźwięk są wręcz zdumiewające, biorąc pod uwagę surowe lokacje produkcji.",
    "(PL) Hipnotyzujące kreacje głównych aktorów trzymają film w ryzach i nie pozwalają widzowi oderwać wzroku.",
    "(PL) Nicks, jakby niepewny, co ma rozśmieszyć ludzi, przechodzi od czerstwej parodii przez rubaszne seksualne dowcipy aż po schematyczną komedię romantyczną.",
    "(PL) Obsada wnosi ogromną energię, a poczucie zabawy i ekscytacji wydaje się tu bardzo na miejscu.",
    "(PL) Czuły, pełen serca dramat rodzinny.",
]

print("EN examples:")
for i, t in enumerate(en_texts):
    print(i, t)

print("\nPL examples (fill them):")
for i, t in enumerate(pl_texts):
    print(i, t)

EN examples:
0 it 's a charming and often affecting journey . 
1 the acting , costumes , music , cinematography and sound are all astounding given the production 's austere locales . 
2 the mesmerizing performances of the leads keep the film grounded and keep the audience riveted . 
3 nicks , seemingly uncertain what 's going to make people laugh , runs the gamut from stale parody to raunchy sex gags to formula romantic comedy . 
4 there 's ... tremendous energy from the cast , a sense of playfulness and excitement that seems appropriate . 
5 a tender , heartfelt family drama . 

PL examples (fill them):
0 (PL) To urocza i często poruszająca podróż.
1 (PL) Gra aktorska, kostiumy, muzyka, zdjęcia i dźwięk są wręcz zdumiewające, biorąc pod uwagę surowe lokacje produkcji.
2 (PL) Hipnotyzujące kreacje głównych aktorów trzymają film w ryzach i nie pozwalają widzowi oderwać wzroku.
3 (PL) Nicks, jakby niepewny, co ma rozśmieszyć ludzi, przechodzi od czerstwej parodii przez rubaszne seksualne

In [12]:
pl_preds, pl_probs = predict_texts(pl_texts)

for en, pl, ep, pp, epr, ppr in zip(en_texts, pl_texts, en_preds, pl_preds, en_probs, pl_probs):
    print("EN:", en)
    print("  pred:", id2label[int(ep)], " prob:", {id2label[i]: float(epr[i]) for i in range(2)})
    print("PL:", pl)
    print("  pred:", id2label[int(pp)], " prob:", {id2label[i]: float(ppr[i]) for i in range(2)})
    print("---")

EN: it 's a charming and often affecting journey . 
  pred: positive  prob: {'negative': 0.06462125480175018, 'positive': 0.9353787899017334}
PL: (PL) To urocza i często poruszająca podróż.
  pred: positive  prob: {'negative': 0.40200600028038025, 'positive': 0.5979939699172974}
---
EN: the acting , costumes , music , cinematography and sound are all astounding given the production 's austere locales . 
  pred: positive  prob: {'negative': 0.14902694523334503, 'positive': 0.8509730100631714}
PL: (PL) Gra aktorska, kostiumy, muzyka, zdjęcia i dźwięk są wręcz zdumiewające, biorąc pod uwagę surowe lokacje produkcji.
  pred: negative  prob: {'negative': 0.5548352599143982, 'positive': 0.4451647400856018}
---
EN: the mesmerizing performances of the leads keep the film grounded and keep the audience riveted . 
  pred: positive  prob: {'negative': 0.07739045470952988, 'positive': 0.9226095676422119}
PL: (PL) Hipnotyzujące kreacje głównych aktorów trzymają film w ryzach i nie pozwalają widzowi

## Wnioski

**Zgodne predykcje : 3/6**
- idx 0: positive → positive, PL dużo mniej pewny: 0.60 vs 0.94
- idx 3: negative → negative, ale prawie remis (PL: 0.51/0.49)
- idx 4: positive → positive, PL znowu mniej pewny: 0.62 vs 0.94

**Niezgodne predykcje 3/6**
idx 1: positive → negative, PL: 0.55 neg / 0.45 pos
idx 2: positive → negative, PL: 0.67 neg
idx 5: positive → negative, PL: 0.53 neg / 0.47 pos

Wyniki są logiczne dla transferu zero-shot w mBERT po fine-tuningu na angielskim: dla EN promptów model jest pewny i dobrze wykrywa sentyment, a po polsku prawie zawsze jest niepewny. Fine-tuning wyłącznie na angielskim przesuwa reprezentacje pod angielski. Nawet gdy mBERT jest wielojęzyczny, podczas fine-tuningu klasyfikator uczy się rozróżniać sentyment na podstawie cech typowych dla angielskiego.

Aby poprawić wyniki można spróbować zwiększyć ilość epoch dla fine-tuningu na EN i dodać więcej danych EN